# Project Cycle 3: Data Check & Cleaning

**Dataset:** `YRBS_2007.csv` | **Target Audience:** US High School Students  
**Responsible Person:** 李宥宣
---

### Team Members

| Name | Student ID |
| :--- | :--- |
| **李宥宣** | `113370237` |
| **許皓崴** | `112370236` |

### Objective
The primary goal of this notebook is to perform **Data Cleaning** and **Variable Recoding** using the approved variables for Cycle 3. We focus on:
1. **Loading** the raw dataset without modification.
2. **Extracting** the approved variables for Question 3 (`WhatIsYourSex` and `SadOrHopeless`).
3. **Handling** missing or invalid values.
4. **Recoding** the response variable according to the required rules.
5. **Exporting** the processed data for reproducibility.

### Research Questions & Variable Definitions

Following the Cycle 3 project guidelines, we selected Question 3 and established our variables:

**Two-Sample Proportion Analysis**
* **Research Question:** Is the proportion of students who felt sad or hopeless different between male and female students?
* **Group Variable:** `WhatIsYourSex` (Gender)
* **Response Variable:** `SadOrHopeless` (Felt sad or hopeless)
* **Recoding Rule for Response Variable:**
  * **Success (1):** code 1 (Yes, felt sad or hopeless) 
  * **Failure (0):** code 2 (No, did not feel sad or hopeless)

### Data Processing Pipeline

To ensure **reproducibility**, this cleaning script follows these steps:
1. **Load Data:** Read `../data/raw/YRBS_2007.csv`.
2. **Subset:** Extract only `WhatIsYourSex` and `SadOrHopeless`.
3. **Handle Missing Data:** Count and drop rows with `NaN`.
4. **Recoding:** Convert `SadOrHopeless` into a binary variable (`SadOrHopeless_Recoded`) strictly following the success=1, failure=2 rule.
5. **Data Validation:** Check the summary statistics and sample sizes of the two groups (Male vs Female).
6. **Export:** Save the final clean dataset to `../data/processed/q3_cleaned_data.csv` for the inference phase.

In [8]:
# Import pandas for data manipulation and analysis
import pandas as pd
# Import numpy for numerical operations, specifically handling NaN (Not a Number) values
import numpy as np
# Import os module to interact with the operating system, like creating directories
import os

# ==========================================================
# 1. Environment Setup and Loading
# ==========================================================
# Define the relative file path for the original, untouched dataset
raw_data_path = '../data/raw/YRBS_2007.csv'
# Define the relative file path where the final cleaned dataset will be exported
processed_data_path = '../data/processed/q3_cleaned_data.csv'

# Create the 'processed' directory if it doesn't exist to prevent FileNotFoundError
# 'exist_ok=True' ensures no error is thrown if the folder is already there
os.makedirs('../data/processed', exist_ok=True)

# Load the raw CSV file into a pandas DataFrame
df_raw = pd.read_csv(raw_data_path)
# Calculate the total number of rows in the raw dataset to track sample size changes
initial_count = len(df_raw)
# Print a confirmation message showing the initial number of participants
print(f"✅ Raw data loaded successfully. Initial sample size: {initial_count}")

# ==========================================================
# 2. Variable Selection (Question 3)
# ==========================================================
# Define the independent variable (group definition) representing gender
group_var = 'WhatIsYourSex'
# Define the dependent variable (response variable) representing feeling sad or hopeless
response_var = 'SadOrHopeless'

# Create a new DataFrame containing only the two required columns for Question 3
# .copy() is used to create an independent object and avoid the 'SettingWithCopyWarning' later
df_selected = df_raw[[group_var, response_var]].copy()

# ==========================================================
# 3. Handling Missing Values (NaN)
# ==========================================================
# Print the title for the missing value statistics section
print("\n[Step 1] Missing Value Statistics:")
# Calculate and print the total number of missing values (NaN) in each selected column
print(df_selected.isnull().sum().to_string())

# Drop any rows that contain missing values in either gender or sadness columns
# We use .copy() again to secure the cleaned data into a new DataFrame
df_cleaned = df_selected.dropna().copy()
# Calculate the remaining sample size after removing participants with incomplete answers
after_dropna_count = len(df_cleaned)
# Print the updated sample size
print(f"\n[Step 2] Sample size after dropping missing values: {after_dropna_count}")

# ==========================================================
# 4. Variable Recoding (SadOrHopeless)
# ==========================================================
# Define a custom function to recode the survey responses into a strict binary format (1 or 0)
def recode_sadness(code):
    # Required rule from instructions: success = code 1, failure = code 2
    if code == 1:
        return 1  # Success: Participant answered 'Yes' (Felt sad or hopeless)
    elif code == 2:
        return 0  # Failure: Participant answered 'No' (Did not feel sad or hopeless)
    else:
        return np.nan # Invalid or unexpected codes are converted to NaN for safe removal

# Define the name for the new recoded column
new_col_name = 'SadOrHopeless_Recoded'
# Apply the recode function to every value in the original response variable column
# and store the result in the newly created column
df_cleaned[new_col_name] = df_cleaned[response_var].apply(recode_sadness)

# Drop any rows that became NaN during the recoding process (e.g., if someone answered '3')
df_final = df_cleaned.dropna().copy()
# Calculate the final, absolute sample size ready for statistical inference
final_count = len(df_final)

# Print verification to ensure recoding was successful and only 0s and 1s remain
print(f"\n[Step 3] Recoding Verification ({new_col_name}):")
# Map the 0 and 1 values to clear readable labels for the output
sadness_labels = {1: '1 (Felt Sad/Hopeless)', 0: '0 (No Sadness)'}
print(df_final[new_col_name].value_counts(dropna=False).rename(sadness_labels).to_string())

# ==========================================================
# 5. Data Validation & Summary (Group Definition)
# ==========================================================
# Print a check for the group variable to observe the gender distribution
print("\n[Step 4] Group Variable Check ('WhatIsYourSex'):")
# Map the 1.0 and 2.0 values to Female and Male for clearer output
gender_labels = {1.0: '1.0 (Female)', 2.0: '2.0 (Male)'}
print(df_final[group_var].value_counts(dropna=False).rename(gender_labels).to_string())

# Print a visual separator and the final project summary report
print("\n" + "="*50)
print(f"🏆 Final Data Cleaning Summary:")
print(f"   - Initial raw samples: {initial_count}")
print(f"   - Final usable samples: {final_count}")
print(f"   - Total attrition rate: {((initial_count - final_count) / initial_count * 100):.2f}%")
print("-" * 50)
print(f"   📊 Group Breakdown (Ready for Z-test):")
print(f"      * Female Students: {df_final[group_var].value_counts().get(1.0, 0)}")
print(f"      * Male Students:   {df_final[group_var].value_counts().get(2.0, 0)}")
print("="*50)

# Export the final, completely clean DataFrame to a CSV file for Student B to use
# 'index=False' prevents pandas from writing the row numbers into the file
df_final.to_csv(processed_data_path, index=False)
# Confirm that the file has been successfully saved to the specified path
print(f"\n💾 Processed dataset saved to: {processed_data_path}")

✅ Raw data loaded successfully. Initial sample size: 14041

[Step 1] Missing Value Statistics:
WhatIsYourSex     13
SadOrHopeless    196

[Step 2] Sample size after dropping missing values: 13833

[Step 3] Recoding Verification (SadOrHopeless_Recoded):
SadOrHopeless_Recoded
0 (No Sadness)           9686
1 (Felt Sad/Hopeless)    4147

[Step 4] Group Variable Check ('WhatIsYourSex'):
WhatIsYourSex
1.0 (Female)    6940
2.0 (Male)      6893

🏆 Final Data Cleaning Summary:
   - Initial raw samples: 14041
   - Final usable samples: 13833
   - Total attrition rate: 1.48%
--------------------------------------------------
   📊 Group Breakdown (Ready for Z-test):
      * Female Students: 6940
      * Male Students:   6893

💾 Processed dataset saved to: ../data/processed/q3_cleaned_data.csv
